In [5]:
%pip install catboost lightgbm optuna

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px # one-liner charts, high-level
import plotly.graph_objects as go # full control chart

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.base import clone

from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier, Pool

import optuna

import typing

import warnings

In [7]:
TARGET = "Will_Buy_EV"
N_SPLITS = 5

def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

cat_columns = [
    # "Gender",
    "City_Type",
    "Current_Car_Type"
    # "Home_Charging_Possible",
    # "Subsidy_Available",
    # "Range_Anxiety_Level",
]
num_columns = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

yes_no_columns = ["Home_Charging_Possible", "Subsidy_Available"]
ordinal_columns = ["Range_Anxiety_Level"]
non_relevant_columns = ['id', 'source']

DRIVE_FOLDER_NAME = "EV Purchases"

In [8]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/train.csv')
    test_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'

  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

In [9]:
# def load_data():
#     df_train = pd.read_csv("train.csv")
#     df_test = pd.read_csv("test.csv")

#     df_test["source"] = "test"
#     df_train["source"] = "train"

#     df = pd.concat([df_train, df_test], ignore_index=True)

#     df[TARGET] = df[TARGET].str.lower().map({"no": 0, "yes": 1})
#     return df

In [10]:
def data_splitter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

In [11]:
train_df, test_df, df = load_data(ENVIRONMENT)

Mounted at /content/drive


In [12]:
df.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV,source
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No,train
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No,train
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes,train
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No,train
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No,train


# EDA

## Info

In [0]:
df.info()

In [0]:
df.describe().T

In [0]:
df.isna().sum()

In [0]:
df.head()

## Adversarial Validation

See if the distributions of train and test sets match

In [0]:
X_adv = df.drop(columns=[TARGET, "source", "id"])
y_adv = df["source"].map({"train": 0, "test": 1})

for c in cat_columns:
    X_adv[c] = X_adv[c].astype("category")

cv_adv = StratifiedKFold(5, shuffle=True, random_state=42)
oof_adv = cross_val_predict(
    LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1),
    X_adv,
    y_adv,
    cv=cv_adv,
    method="predict_proba",
)[:, 1]
print(roc_auc_score(y_adv, oof_adv))

## EDA on Features

In [0]:
# Income to Target
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x='Annual_Income_USD',
    hue=TARGET,
    multiple='fill',
    kde=True,
    bins=30
)

In [0]:
df[df['Annual_Income_USD'].between(31_004, 41_970)]['Will_Buy_EV'].unique()

In [0]:
df[df['source'] == 'train'].groupby('Annual_Income_USD')['Will_Buy_EV'].agg(rate='mean', n='size').sort_index()

In [0]:
df.groupby('Annual_Income_USD')['Will_Buy_EV'].value_counts(normalize=True).reset_index().query('proportion in [0, 1]')

In [0]:
# Income to Target
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x='Annual_Income_USD',
    hue=TARGET,
    # multiple='fill',
    kde=True,
    bins=30
)

In [0]:
# Daily Commute to Target
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x='Daily_Commute_km',
    hue=TARGET,
    multiple='fill',
    kde=True,
    bins=18
)

### Corelations

In [0]:
z = df[df['source']=='train'].copy()

z[['Home_Charging_Possible', 'Subsidy_Available']] = z[['Home_Charging_Possible', 'Subsidy_Available']].apply(lambda col: col.str.lower().map({'yes': 1, 'no': 0}))

z['Range_Anxiety_Level'] = z['Range_Anxiety_Level'].map({'Low':0, 'Medium':1, 'High': 2})

z = pd.concat([z, pd.get_dummies(df['Gender'], prefix='Gender').astype(int)], axis=1)
z.drop(columns=['Gender'], inplace=True)

z = pd.concat([z, pd.get_dummies(df['Current_Car_Type'], prefix='Current_Car_Type').astype(int)], axis=1)
z.drop(columns=['Current_Car_Type'], inplace=True)

z = pd.concat([z, pd.get_dummies(df['City_Type'], prefix='City_Type').astype(int)], axis=1)
z.drop(columns=['City_Type'], inplace=True)

In [0]:
z_corr = z.corr(numeric_only=True)
corr_sort_cols = z_corr['Will_Buy_EV'].abs().sort_values(ascending=False).index.to_list()
z_corr_sorted = z_corr.loc[corr_sort_cols, corr_sort_cols]
z_corr_mask = np.triu(np.ones_like(z_corr_sorted, dtype=bool), k=1)
z_corr_sorted[z_corr_mask] = np.nan

px.imshow(
    z_corr_sorted,
    aspect=True,
    color_continuous_scale="RdBu_r",
    text_auto='.2f'
).update_traces(hoverongaps=True).update_layout(width=1200, height=500)

### Environmental Concern Level

In [0]:
px.histogram(
    df,
    x='Environmental_Concern_Level',
    color='Will_Buy_EV',
    barmode='overlay',
    marginal='box',
    opacity=0.6
)

In [0]:
px.histogram(
    df,
    x='Range_Anxiety_Level',
    color='Will_Buy_EV',
    barmode='overlay',
    # marginal='box',
    opacity=0.6
)

In [0]:
df

In [0]:
fig = px.histogram(
    df,
    x='Annual_Income_USD',
    color='Will_Buy_EV',
    barmode='overlay',
    marginal='box',
    opacity=0.6,
    nbins=10
)

displayHTML(fig.to_html(include_plotlyjs='cdn'))

# Feature Engineering

In [0]:
df.head()

In [13]:
yes_no_columns

['Home_Charging_Possible', 'Subsidy_Available']

In [20]:
def feature_engineering(df):

    for c in df[yes_no_columns]:
        df[c] = df[c].str.lower().map({"no": 0, "yes": 1})


    df["Range_Anxiety_Level"] = (
        df["Range_Anxiety_Level"].str.lower().map({"low": 0, "medium": 1, "high": 2})
    )

    df['Total_Charging_Stations'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work'] + df['Home_Charging_Possible']

    # df['Km_Per_Station'] = df['Daily_Commute_km'] / df['Total_Charging_Stations'].replace(0, np.nan)
    # df['No_Charging'] = (df['Total_Charging_Stations'] == 0).astype('int8')

    df['Realistic_Concern'] = df['Environmental_Concern_Level'] - df['Range_Anxiety_Level']
    
    df["Anxiety_Environmental"] = (
    df["Range_Anxiety_Level"].astype(str)
    + "_"
    + df["Environmental_Concern_Level"].astype(str))

    df['Anxiety_Commute_Ratio'] = np.where(
        df['Daily_Commute_km'] == 0,
        0,
        df['Range_Anxiety_Level'] / df['Daily_Commute_km']
    )

    df["High_Income_EV_Potential"] = (df["Annual_Income_USD"] >= 170537.0).astype('int8')
    df['income_per_car'] = df['Annual_Income_USD'] / (df['Number_of_Cars_Owned'] + 1)

    # # The Dead Zone (0% buy rate region)
    # df['is_dead_zone'] = ((df['Annual_Income_USD'] >= 38000.0) & (df['Annual_Income_USD'] <= 42000.0)).astype('int8')

    # Subsidy combinations

    df['subsidy_income'] = df['Annual_Income_USD'] * df['Subsidy_Available']
    df['subsidy_concern'] = df['Environmental_Concern_Level'] * df['Subsidy_Available']
    df['home_charging_subsidy'] = df['Home_Charging_Possible'] * df['Subsidy_Available']

    df = df.drop(columns='Gender')

    # num_cols = [
    #     'Age', 'Annual_Income_USD', 'Daily_Commute_km',
    #     'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
    #     'Charging_Stations_Near_Work', 'Environmental_Concern_Level'
    # ]

    # # Digit Decomposition
    # for c in num_cols:
    #     for k in range(-4, 4):
    #         df[f'{c}_digit{k}'] = (df[c].fillna(0) // (10.0 ** k) % 10).astype('int8')

    return df

In [0]:
# def frequency_encoding(df, columns=[]):
#     if len(columns) == 0:
#         pass
#     else:
#         # Frequency Encoding
#         for col in columns:
#             mapper = df[col].value_counts(normalize=True)
#             df[f'{col}_freq'] = df[col].map(mapper)
            
#     return df

# Baseline model (Logistic Regression)

In [0]:
df.head()

In [0]:
X, y, submission_set = data_splitter(df)

In [0]:
log_reg_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_columns),
        ('scaler', StandardScaler(), num_columns),
    ],
    remainder="drop"
)

log_reg_pipeline = Pipeline([
    ('preprocessor', log_reg_preprocessor),
    ('logreg', LogisticRegression(max_iter=1000))
])

In [0]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    random_state=42,
    shuffle=True
)

In [0]:
oof = np.zeros(len(X))
submission_results = np.zeros(len(submission_set))
folds_train_results, folds_validation_results = [], []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
    X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    log_reg_pipeline.fit(X_train, y_train)
    oof_pred = log_reg_pipeline.predict_proba(X_val)[:,1]
    train_pred = log_reg_pipeline.predict_proba(X_train)[:,1]
    
    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    folds_train_results.append(train_auc)
    folds_validation_results.append(oof_auc)
    submission_results += log_reg_pipeline.predict_proba(submission_set)[:,1] / N_SPLITS

    print(f"=======FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc} | TEST AUC: {oof_auc} | DELTA {oof_auc - train_auc}")

tr, va = np.array(folds_train_results), np.array(folds_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

## Baseline ROC AUC

TRAIN AVG AUC: 0.93811 +/- 0.00020
VAL AVG AUC: 0.93810 +/- 0.00081

OOF AUC: 0.93809

# XGBoost

In [0]:
df = load_data()
df_ = feature_engineering(df)
X, y, submission_set = data_splitter(df_)

xgb_params = {
    "n_estimators": 1095,
    "max_depth": 8,
    "learning_rate": 0.06757009447016467,
    "subsample": 0.8526064253616809,
    "colsample_bytree": 0.713964646305348,
    "min_child_weight": 39,
    "reg_alpha": 7.741721190554106e-05,
    "reg_lambda": 0.009405232270558986,
    "gamma": 0.6884035558163024,
}

xgb_model = XGBClassifier(**xgb_params, random_state=42, n_jobs=1)

xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_columns),
    ],
    remainder="passthrough",
)

xgb_pipeline = Pipeline([("preprocessor", xgb_preprocessor), ("xgb", xgb_model)])

In [0]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    random_state=42,
    shuffle=True
)

oof = np.zeros(len(X))
submission_results = np.zeros(len(submission_set))
folds_train_results, folds_validation_results = [], []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
    X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    xgb_pipeline.fit(X_train, y_train)
    oof_pred = xgb_pipeline.predict_proba(X_val)[:,1]
    train_pred = xgb_pipeline.predict_proba(X_train)[:,1]
    
    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    folds_train_results.append(train_auc)
    folds_validation_results.append(oof_auc)
    submission_results += xgb_pipeline.predict_proba(submission_set)[:,1] / N_SPLITS

    print(f"=======FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc} | TEST AUC: {oof_auc} | DELTA {oof_auc - train_auc}")

tr, va = np.array(folds_train_results), np.array(folds_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

# LightGBM

In [28]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    random_state=42,
    shuffle=True
)

In [17]:
_, _, df = load_data(ENVIRONMENT)
df = feature_engineering(df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
df.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,...,source,Total_Charging_Stations,Realistic_Concern,Anxiety_Environmental,Anxiety_Commute_Ratio,High_Income_EV_Potential,income_per_car,subsidy_income,subsidy_concern,home_charging_subsidy
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,...,train,11,1.0,0_1.0,0.0,0,30962.333333,0.0,0.0,0
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,...,train,5,4.0,0_4.0,0.0,0,15000.000000,0.0,0.0,0
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,...,train,23,5.0,0_5.0,0.0,0,47194.500000,94389.0,5.0,0
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,...,train,16,3.0,0_3.0,0.0,0,24526.666667,0.0,0.0,0
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,...,train,6,3.0,0_3.0,0.0,0,28949.000000,0.0,0.0,0


In [21]:
_, _, df = load_data(ENVIRONMENT)
df = feature_engineering(df)
# frequency_encoding_columns = df.drop(columns=['id', 'source', 'Will_Buy_EV']).columns.to_list()
df[['City_Type','Current_Car_Type', 'Anxiety_Environmental']] = df[['City_Type','Current_Car_Type', 'Anxiety_Environmental']].astype('category')
# df = frequency_encoding(df, frequency_encoding_columns)
X, y, submission_df = data_splitter(df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
df.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,City_Type,Current_Car_Type,...,source,Total_Charging_Stations,Realistic_Concern,Anxiety_Environmental,Anxiety_Commute_Ratio,High_Income_EV_Potential,income_per_car,subsidy_income,subsidy_concern,home_charging_subsidy
0,0,66,92887.0,23.4,2,3,7,1.0,Suburban,Sedan,...,train,11,1.0,0_1.0,0.0,0,30962.333333,0.0,0.0,0
1,1,38,30000.0,5.0,1,2,2,4.0,Rural,SUV,...,train,5,4.0,0_4.0,0.0,0,15000.000000,0.0,0.0,0
2,2,26,94389.0,36.8,1,8,15,5.0,Urban,Sedan,...,train,23,5.0,0_5.0,0.0,0,47194.500000,94389.0,5.0,0
3,3,66,73580.0,23.7,2,6,9,3.0,Suburban,Hatchback,...,train,16,3.0,0_3.0,0.0,0,24526.666667,0.0,0.0,0
4,4,54,57898.0,50.8,1,2,3,3.0,Suburban,Hatchback,...,train,6,3.0,0_3.0,0.0,0,28949.000000,0.0,0.0,0


In [23]:
submission_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 286571 entries, 668665 to 955235
Data columns (total 22 columns):
 #   Column                       Non-Null Count   Dtype   
---  ------                       --------------   -----   
 0   id                           286571 non-null  int64   
 1   Age                          286571 non-null  int64   
 2   Annual_Income_USD            286571 non-null  float64 
 3   Daily_Commute_km             286571 non-null  float64 
 4   Number_of_Cars_Owned         286571 non-null  int64   
 5   Charging_Stations_Near_Home  286571 non-null  int64   
 6   Charging_Stations_Near_Work  286571 non-null  int64   
 7   Environmental_Concern_Level  286571 non-null  float64 
 8   City_Type                    286571 non-null  category
 9   Current_Car_Type             286571 non-null  category
 10  Home_Charging_Possible       286571 non-null  int64   
 11  Subsidy_Available            286571 non-null  int64   
 12  Range_Anxiety_Level          286571 non-null

In [0]:
def objective_lgb(trial):
    max_depth = trial.suggest_int("max_depth", 4, 12)
    params = {
        "n_estimators": 3000,          # fixed; early stopping decides the real count
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, min(255, 2 ** max_depth)),
        "max_depth": max_depth,
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    oof_local = np.zeros(len(X))
    iters = []

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
            X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
            X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]


            model = LGBMClassifier(**params)
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                categorical_feature=['City_Type','Current_Car_Type', 'Anxiety_Environmental'],
                eval_metric="auc",
                callbacks=[early_stopping(100, verbose=False)],
            )

            oof_local[val_idx] = model.predict_proba(X_val)[:, 1]
            iters.append(model.best_iteration_)

            # let Optuna kill hopeless trials early
            trial.report(roc_auc_score(y_val, oof_local[val_idx]), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()


    trial.set_user_attr("best_iters", iters)
    return roc_auc_score(y, oof_local)
# Re-split X and y to include the newly engineered features
# X, y, submission_df = data_spliter(df)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10 ,n_warmup_steps=2),
)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective_lgb, n_trials=100, show_progress_bar=True)

print(study.best_value)
print(study.best_params)
# how much of the top is noise?
print(study.trials_dataframe().sort_values("value", ascending=False)["value"].head(15).to_string())

In [0]:
print(study.best_value)
print(study.best_params)

In [24]:
oof = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
submission_results = np.zeros(len(submission_df))
submission_results_lgb = np.zeros(len(submission_df))
fold_train_results, fold_validation_results = [], []
best_iters = []

X_sub_raw = submission_df.drop(columns=["id"])

In [25]:
# Columns in submission_df but not in X
set(X_sub_raw.columns) - set(X.columns)

set()

In [26]:
lgb_test_params = {'n_estimators': 3000, 'max_depth': 4, 'learning_rate': 0.03755714668175416, 'num_leaves': 16, 'min_child_samples': 118, 'subsample': 0.8960672366371799, 'subsample_freq': 5, 'colsample_bytree': 0.5117178113431882, 'reg_alpha': 5.7764356817014716e-08, 'reg_lambda': 1.5042296595462026e-08}

lgb_optuna_params_1 = {'n_estimators': 3000, 'max_depth': 4, 'learning_rate': 0.03597271512159211, 'num_leaves': 16, 'min_child_samples': 193, 'subsample': 0.9146558054022962, 'subsample_freq': 1, 'colsample_bytree': 0.5675296747189175, 'reg_alpha': 6.202845357521301e-08, 'reg_lambda': 0.00015263204457498}

## Full Training

In [30]:
for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
  with warnings.catch_warnings():
    print(f"Fold started: {fold}\n")    
    warnings.simplefilter("ignore")
    X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]


    model = LGBMClassifier(**lgb_optuna_params_1, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=['City_Type','Current_Car_Type', 'Anxiety_Environmental'],
        eval_metric="auc",
        callbacks=[early_stopping(100, verbose=False)],
    )

    oof_pred = model.predict_proba(X_val)[:, 1]
    train_pred = model.predict_proba(X_train)[:, 1]

    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    oof_lgb[val_idx] = oof_pred
    fold_train_results.append(train_auc)
    fold_validation_results.append(oof_auc)
    best_iters.append(model.best_iteration_)

    submission_results += model.predict_proba(X_sub_raw)[:, 1] / N_SPLITS
    submission_results_lgb += model.predict_proba(X_sub_raw)[:, 1] / N_SPLITS

    print(f"======= FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc:.5f} | VAL AUC: {oof_auc:.5f} | "
          f"DELTA: {oof_auc - train_auc:+.5f} | BEST ITER: {model.best_iteration_}")

tr, va = np.array(fold_train_results), np.array(fold_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | "
      f"VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"MEAN BEST ITER: {np.mean(best_iters):.0f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

Fold started: 0

======= FOLD: 0 =======
TRAIN AUC: 0.94557 | VAL AUC: 0.94107 | DELTA: -0.00449 | BEST ITER: 1876
Fold started: 1

======= FOLD: 1 =======
TRAIN AUC: 0.94534 | VAL AUC: 0.94214 | DELTA: -0.00320 | BEST ITER: 1741
Fold started: 2

======= FOLD: 2 =======
TRAIN AUC: 0.94553 | VAL AUC: 0.94349 | DELTA: -0.00204 | BEST ITER: 2039
Fold started: 3

======= FOLD: 3 =======
TRAIN AUC: 0.94546 | VAL AUC: 0.94286 | DELTA: -0.00260 | BEST ITER: 1947
Fold started: 4

======= FOLD: 4 =======
TRAIN AUC: 0.94500 | VAL AUC: 0.94233 | DELTA: -0.00268 | BEST ITER: 1625

====== CROSS VALIDATION ENDED ======
TRAIN AVG AUC: 0.94538 +/- 0.00020 | VAL AVG AUC: 0.94238 +/- 0.00080
MEAN BEST ITER: 1846

OOF AUC: 0.94237


In [38]:
final_tree_number = round(np.mean(best_iters) * 1.1, 0).astype(int)

In [41]:
final_params = {k: v for k,v in lgb_optuna_params_1.items() if k != 'n_estimators'}

In [44]:
model_lgb = LGBMClassifier(
    n_estimators=final_tree_number,
    **final_params,
    random_state=42, n_jobs=-1, verbose=-1
)
model_lgb.fit(
    X,
    y,
    categorical_feature=['City_Type','Current_Car_Type', 'Anxiety_Environmental']
)

LGBMClassifier(colsample_bytree=0.5675296747189175,
               learning_rate=0.03597271512159211, max_depth=4,
               min_child_samples=193, n_estimators=np.int64(2030), n_jobs=-1,
               num_leaves=16, random_state=42, reg_alpha=6.202845357521301e-08,
               reg_lambda=0.00015263204457498, subsample=0.9146558054022962,
               subsample_freq=1, verbose=-1)

In [45]:
import joblib

# After training on a DataFrame X_train
bundle = {
    "model": model_lgb,
    "features": list(X.columns),                        # order
    "dtypes": X.dtypes.astype(str).to_dict(),           # type of each feature
}
joblib.dump(bundle, "model.joblib")

['model.joblib']

In [0]:
model_name = "lgbm"

submission_df_lgb = submission_df.copy()
training_lgb = X.copy()

submission_df_lgb[TARGET] = submission_results_lgb
training_lgb['pred'] = oof_lgb
lgbm_train_prediction = training_lgb.reset_index()[['index', 'pred']]
lgbm_prediction = submission_df_lgb[['id', TARGET]]

# Catboost

In [0]:
df = load_data()

In [0]:
df.head()

In [0]:
df = load_data()
df = feature_engineering(df)
frequency_encoding_columns = df.drop(columns=['id', 'source', 'Will_Buy_EV']).columns.to_list()
catboost_categories = [
    'Age', 'Annual_Income_USD', 'Daily_Commute_km',
    'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work', 'Environmental_Concern_Level',
    'Gender','City_Type','Current_Car_Type', 'Anxiety_Environmental', 'Range_Anxiety_Level'
]
df[catboost_categories] = df[catboost_categories].astype('str')
df = frequency_encoding(df, frequency_encoding_columns)
X, y, submission_df = data_splitter(df)

## Optuna

In [0]:
from sklearn.model_selection import train_test_split

X_reduced, _, y_reduced, _ = train_test_split(
    X, y, train_size=100_000, stratify=y, random_state=42
)

In [0]:
skf_opt = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
def objective_catboost(trial):

    params = {
        'iterations': 5000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'depth': trial.suggest_int('depth', 3, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.5, 30.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.0, 5.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 30, 150),
        'eval_metric': 'AUC',
        'early_stopping_rounds': 200,
        'use_best_model': True,
        'allow_writing_files': False,
        'random_state': 42,
        'verbose': 0,
    }

    bt = trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli'])
    params['bootstrap_type'] = bt
    if bt == 'Bayesian':
        params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.0, 2.0)
    else:
        params['subsample'] = trial.suggest_float('subsample', 0.6, 1.0)

    oof_local = np.zeros(len(X_reduced))
    iters = []

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for fold, (tr_idx, val_idx) in enumerate(skf_opt.split(X_reduced, y_reduced)):
            y_train = y_reduced.iloc[tr_idx]
            y_val =  y_reduced.iloc[val_idx]

            train_pool = Pool(X_reduced.iloc[tr_idx], y_train, cat_features=catboost_categories)
            valid_pool = Pool(X_reduced.iloc[val_idx], y_val, cat_features=catboost_categories)


            model = CatBoostClassifier(
                **params,
                task_type='GPU',
                devices='0'
            )
            model.fit(
                train_pool,
                eval_set=valid_pool
            )

            oof_local[val_idx] = model.predict_proba(valid_pool)[:, 1]
            iters.append(model.best_iteration_)

            # let Optuna kill hopeless trials early
            trial.report(roc_auc_score(y_val, oof_local[val_idx]), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()


    trial.set_user_attr("best_iters", iters)
    return roc_auc_score(y_reduced, oof_local)
catboost_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0),
)
optuna.logging.set_verbosity(optuna.logging.WARNING)
catboost_study.optimize(objective_catboost, n_trials=50, show_progress_bar=True, n_jobs=1)

print(catboost_study.best_value)
print(catboost_study.best_params)
# how much of the top is noise?
print(catboost_study.trials_dataframe().sort_values("value", ascending=False)["value"].head(15).to_string())

In [0]:
print(catboost_study.best_value)
print(catboost_study.best_params)

### Reduced Training

In [0]:
oof = np.zeros(len(X_reduced))
submission_results = np.zeros(len(submission_df))
fold_train_results, fold_validation_results = [], []
best_iters = []

X_sub_raw = submission_df.drop(columns=["id"])
sub_pool = Pool(X_sub_raw[X.columns], cat_features=catboost_categories)

catboost_test_params = {
    "iterations": 5000,
    "learning_rate": 0.02,
    "depth": 6,
    "l2_leaf_reg": 13,
    "random_strength": 3.4165471601740314,
#     "min_data_in_leaf": 100,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.9557182534037572,
    "border_count":250,
    "one_hot_max_size":2,
    "use_best_model": True,
    "allow_writing_files": False,
    "random_state": 42,
    "verbose": 0,
}

In [0]:
for fold, (tr_idx, val_idx) in enumerate(cv.split(X_reduced, y_reduced)):
  with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    y_train = y_reduced.iloc[tr_idx]
    y_val =  y_reduced.iloc[val_idx]

    train_pool = Pool(X_reduced.iloc[tr_idx], y_train, cat_features=catboost_categories)
    valid_pool = Pool(X_reduced.iloc[val_idx], y_val, cat_features=catboost_categories)


    model = CatBoostClassifier(
        **catboost_test_params,
        task_type='GPU',
        devices='0',
        eval_metric='AUC',
        early_stopping_rounds=200,
    )
    model.fit(
        train_pool,
        eval_set=valid_pool,
        verbose=200
    )

    oof_pred = model.predict_proba(valid_pool)[:, 1]
    train_pred = model.predict_proba(train_pool)[:, 1]

    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    fold_train_results.append(train_auc)
    fold_validation_results.append(oof_auc)
    best_iters.append(model.best_iteration_)

    submission_results += model.predict_proba(X_sub_raw)[:, 1] / N_SPLITS

    print(f"======= FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc:.5f} | VAL AUC: {oof_auc:.5f} | "
          f"DELTA: {oof_auc - train_auc:+.5f} | BEST ITER: {model.best_iteration_}")

tr, va = np.array(fold_train_results), np.array(fold_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | "
      f"VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"MEAN BEST ITER: {np.mean(best_iters):.0f}")
print(f"\nOOF AUC: {roc_auc_score(y_reduced, oof):.5f}")

## Full Training

In [0]:
oof = np.zeros(len(X))
oof_cb = np.zeros(len(X))
submission_results = np.zeros(len(submission_df))
submission_results_cb = np.zeros(len(submission_df))
fold_train_results, fold_validation_results = [], []
best_iters = []

X_sub_raw = submission_df.drop(columns=["id"])
sub_pool = Pool(X_sub_raw[X.columns], cat_features=catboost_categories)

catboost_test_params = {
    "iterations": 5000,
    "learning_rate": 0.08,
    "depth": 6,
    "l2_leaf_reg": 13,
    "random_strength": 3.4165471601740314,
#     "min_data_in_leaf": 100,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.9557182534037572,
    "border_count":250,
    "one_hot_max_size":2
}

catboost_optuna_1 = {'iterations': 5000, 'learning_rate': 0.0376197979622221, 'depth': 5, 'l2_leaf_reg': 9.056531453750873, 'random_strength': 4.3273337838346055, 'min_data_in_leaf': 117, 'bootstrap_type': 'Bernoulli', 'subsample': 0.9557182534037572}

In [0]:
for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
  with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    y_train = y.iloc[tr_idx]
    y_val =  y.iloc[val_idx]

    train_pool = Pool(X.iloc[tr_idx], y_train, cat_features=catboost_categories)
    valid_pool = Pool(X.iloc[val_idx], y_val, cat_features=catboost_categories)


    model = CatBoostClassifier(
        # **catboost_test_params,
        **catboost_optuna_1,
        task_type='GPU',
        devices='0',
        eval_metric='AUC',
        early_stopping_rounds= 200,
        use_best_model = True,
        allow_writing_files= False,
        # random_state= 42,
        verbose= 0
    )
    model.fit(
        train_pool,
        eval_set=valid_pool,
        verbose=200
    )

    oof_pred = model.predict_proba(valid_pool)[:, 1]
    train_pred = model.predict_proba(train_pool)[:, 1]

    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    oof_cb[val_idx] = oof_pred
    fold_train_results.append(train_auc)
    fold_validation_results.append(oof_auc)
    best_iters.append(model.best_iteration_)

    submission_results += model.predict_proba(X_sub_raw)[:, 1] / N_SPLITS
    submission_results_cb += model.predict_proba(X_sub_raw)[:, 1] / N_SPLITS

    print(f"======= FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc:.5f} | VAL AUC: {oof_auc:.5f} | "
          f"DELTA: {oof_auc - train_auc:+.5f} | BEST ITER: {model.best_iteration_}")

tr, va = np.array(fold_train_results), np.array(fold_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | "
      f"VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"MEAN BEST ITER: {np.mean(best_iters):.0f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

In [0]:
model_name = "catboost"

submission_df_cb = submission_df.copy()
training_cb = X.copy()

submission_df_cb[TARGET] = submission_results_cb
training_cb['pred'] = oof_cb
catboost_train_prediction = training_cb.reset_index()[['index', 'pred']]
catboost_prediction = submission_df_cb[['id', TARGET]]

catboost_train_prediction.to_parquet(
    f"oof_{model_name}.parquet", index=False)
catboost_prediction.to_parquet(
    f"test_{model_name}.parquet", index=False)

In [0]:
# Catboost
# Default metric period is 5 because AUC is/are not implemented for GPU
# ======= FOLD: 0 =======
# TRAIN AUC: 0.94995 | VAL AUC: 0.94469 | DELTA: -0.00525 | BEST ITER: 4479
# ======= FOLD: 1 =======
# TRAIN AUC: 0.94958 | VAL AUC: 0.94515 | DELTA: -0.00443 | BEST ITER: 3800
# ======= FOLD: 2 =======
# TRAIN AUC: 0.94937 | VAL AUC: 0.94646 | DELTA: -0.00292 | BEST ITER: 3765
# ======= FOLD: 3 =======
# TRAIN AUC: 0.94943 | VAL AUC: 0.94587 | DELTA: -0.00356 | BEST ITER: 3089
# ======= FOLD: 4 =======
# TRAIN AUC: 0.94924 | VAL AUC: 0.94543 | DELTA: -0.00381 | BEST ITER: 2618

# ====== CROSS VALIDATION ENDED ======
# TRAIN AVG AUC: 0.94952 +/- 0.00024 | VAL AVG AUC: 0.94552 +/- 0.00060
# MEAN BEST ITER: 3550

# OOF AUC: 0.94551

In [0]:
# LGBM
# ======= FOLD: 0 =======
# TRAIN AUC: 0.95002 | VAL AUC: 0.94366 | DELTA: -0.00636 | BEST ITER: 2221
# ======= FOLD: 1 =======
# TRAIN AUC: 0.94975 | VAL AUC: 0.94451 | DELTA: -0.00524 | BEST ITER: 2173
# ======= FOLD: 2 =======
# TRAIN AUC: 0.95018 | VAL AUC: 0.94572 | DELTA: -0.00446 | BEST ITER: 2476
# ======= FOLD: 3 =======
# TRAIN AUC: 0.94911 | VAL AUC: 0.94493 | DELTA: -0.00418 | BEST ITER: 1945
# ======= FOLD: 4 =======
# TRAIN AUC: 0.94902 | VAL AUC: 0.94472 | DELTA: -0.00430 | BEST ITER: 1863

# ====== CROSS VALIDATION ENDED ======
# TRAIN AVG AUC: 0.94961 +/- 0.00047 | VAL AVG AUC: 0.94471 +/- 0.00067
# MEAN BEST ITER: 2136

# OOF AUC: 0.94470

# Model Blending

In [0]:
lgbm_train_prediction

In [0]:
catboost_train_prediction = pd.read_parquet('oof_catboost.parquet')
catboost_prediction = pd.read_parquet('test_catboost.parquet')

In [0]:
catboost_prediction

In [0]:
oof_full = pd.merge(X, y, left_index=True, right_index=True).reset_index()[['index', TARGET]]

In [0]:
prediction_full  = submission_df.copy()
prediction_full = prediction_full[['id']]

In [0]:
oof_full = oof_full.merge(catboost_train_prediction.rename(columns={"pred": "catboost"}), on="index")
oof_full = oof_full.merge(lgbm_train_prediction.rename(columns={"pred": "lgbm"}), on="index")

In [0]:
prediction = catboost_prediction.rename(columns={"Will_Buy_EV": "catboost"}).merge(lgbm_prediction.rename(columns={"Will_Buy_EV": "lgbm"}), on="id")

In [0]:
prediction

In [0]:
oof_full

In [0]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

y = oof_full["Will_Buy_EV"].values

# Rank-transform so both models are on the same scale (only ordering matters for AUC)
oof_lgb = rankdata(oof_full["lgbm"]) / len(oof_full)
oof_cb  = rankdata(oof_full["catboost"]) / len(oof_full)

test_lgb = rankdata(prediction["lgbm"]) / len(prediction)
test_cb  = rankdata(prediction["catboost"]) / len(prediction)

# Individual scores as a sanity check
print("LGBM AUC:    ", roc_auc_score(y, oof_full["lgbm"]))
print("CatBoost AUC:", roc_auc_score(y, oof_full["catboost"]))
print("Correlation: ", np.corrcoef(oof_full["lgbm"], oof_full["catboost"])[0, 1])

# Search the weight on LGBM (CatBoost gets 1 - w)
best_w, best_score = 0, 0
for w in np.linspace(0, 1, 101):
    score = roc_auc_score(y, w * oof_lgb + (1 - w) * oof_cb)
    if score > best_score:
        best_w, best_score = w, score

print(f"Best weight on LGBM: {best_w:.2f}  |  Blend AUC: {best_score:.5f}")

# Build the submission
submission = pd.DataFrame({
    "id": prediction["id"],
    "Will_Buy_EV": best_w * test_lgb + (1 - best_w) * test_cb,
})
submission.to_csv("blend_catboost_lgbm_v1.csv", index=False)

In [0]:
models = ["lgbm_v1", "catboost_v1"]
path = "/kaggle/input/..."  # adjust

oof = train[["id", "target"]].copy()
sub = test[["id"]].copy()
for m in models:
    oof = oof.merge(pd.read_parquet(f"{path}/oof_{m}.parquet")
                    .rename(columns={"pred": m}), on="id")
    sub = sub.merge(pd.read_parquet(f"{path}/test_{m}.parquet")
                    .rename(columns={"pred": m}), on="id")

assert len(oof) == len(train) and len(sub) == len(test)

# Submission

In [0]:
submission_df[TARGET] = submission_results
lgb_prediction = submission_df[['id', TARGET]]
# lgb_prediction.to_csv('lgb_prediction_v7.csv', index=False)

In [0]:
submission_df[TARGET] = submission_results
catboost_prediction = submission_df[['id', TARGET]]
catboost_prediction.to_csv('catboost_prediction_v4.csv', index=False)

In [0]:
submit_df = submission_set.copy()
submit_df[TARGET] = submission_results

In [0]:
submit_df[['id', TARGET]].to_csv('xgb_submission_v1.csv', index=False)

In [0]:
submit_df[['id', TARGET]]